# Marco 4: modelos M1 em sessão única (T-4.3b, D-53, D-64)

Notebook sem lógica: monta o Drive, carrega o config, confere o ambiente e dispara a sessão em segundo plano (`python -m dengue_sinan.runner`). O trabalho todo fica em `src/dengue_sinan/runner.py`.

Pasta do projeto no Drive: `MyDrive/dengue-comorbidity-risk/`. Ajuste `PASTA_DRIVE` se for outra.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# Código no GitHub, commitado pelo orquestrador antes desta execução (D-66).
# Ambiente a partir do lock (NFR-02): o runtime precisa ser Python 3.13.
import os
import sys

assert sys.version_info[:2] == (3, 13), (
    f'Python {sys.version_info[0]}.{sys.version_info[1]} aqui; o projeto usa 3.13. '
    'Se falhar, escolha o runtime 2026.10 em "Past runtime versions" e rode de novo.'
)

REPO_URL = 'https://github.com/PabloThePmX/dengue-comorbidity-risk'
PASTA_REPO = '/content/dengue-comorbidity-risk'

if not os.path.isdir(PASTA_REPO):
    !git clone -q {REPO_URL} {PASTA_REPO}
%cd {PASTA_REPO}
!git pull -q
!pip install -q uv
# -q e saída para /dev/null: o lock vai para o arquivo (-o), sem imprimir as linhas na célula
!uv export --frozen --no-hashes --no-dev --no-emit-project \
    -q -o /content/requirements-lock.txt > /dev/null
lock = '/content/requirements-lock.txt'
assert os.path.getsize(lock) > 0, 'uv export não gerou o lock'
!pip install -q -r /content/requirements-lock.txt
!pip install -q -e . --no-deps
# O kernel só enxerga o pacote editável depois de reiniciar: o src entra no path desta sessão.
sys.path.insert(0, f'{PASTA_REPO}/src')

In [ ]:
from dengue_sinan import runner
from dengue_sinan.config import load_config

PASTA_DRIVE = '/content/drive/MyDrive/dengue-comorbidity-risk'

cfg = load_config()  # config/base.yaml do clone
config_sessao = runner.config_da_sessao(cfg, PASTA_DRIVE)  # dados e mlruns no Drive
LOG = f'{PASTA_DRIVE}/logs/sessao-m1.log'
print('config da sessão:', config_sessao)


In [ ]:
# TabFM (D-69, D-90): instalado só nesta sessão do Colab, fora do lock.
# Vem da main do GitHub em commit fixo (tabfm.commit_pacote); o PyPI só tem a JAX.
# --no-deps: as dependências vêm do lock; só o que o lock não cobre é listado.
# O extra [pytorch] só pede o torch, que o lock já tem.
# Esta célula só instala; não há lógica de treino aqui.
commit_tabfm = cfg['tabfm']['commit_pacote']
pacotes = (f'"git+https://github.com/google-research/tabfm.git@{commit_tabfm}" '
           'absl-py "jaxtyping<0.3" "typeguard<3" huggingface-hub')
!pip install -q --no-deps {pacotes}

In [ ]:
# Checagem do ambiente de instalação (sem lógica de treino).
# Se torch ou numpy já estavam importados antes dos pip install acima,
# reinicie o runtime uma vez (Ambiente de execução > Reiniciar sessão) e rode de novo
# a partir da célula de clone: o processo antigo continua com as versões antigas.
import torch

print('torch', torch.__version__)
print('cuda do torch', torch.version.cuda)
print('cuda disponível', torch.cuda.is_available())
print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma')
!nvidia-smi
!pip check

In [ ]:
# Confere GPU e RAM contra cfg.ambiente antes de começar (D-64).
from dengue_sinan import verify

for check in verify.environment(load_config(config_sessao)):
    print(check)

> **Aviso:** nunca use *Interromper execução* nem *Reiniciar sessão* com o runner vivo. O Interromper mata o processo do runner: ele retoma do SQLite do Optuna no próximo disparo, mas o ajuste em curso se perde. Para ver se ainda roda, use `pgrep -f dengue_sinan.runner` ou a célula de acompanhamento.

In [ ]:
# Segundo plano (T-4.3b, D-53): o runner roda num processo próprio, com log no Drive.
# Não use `!... &`: o `&` segura a saída desta célula, e o Interromper mata o runner.
import subprocess

# o runner não pode estar vivo: dois ao mesmo tempo disputam o SQLite do Optuna
vivos = subprocess.run(['pgrep', '-f', 'dengue_sinan.runner'],
                       capture_output=True, text=True).stdout.split()
assert not vivos, f'runner já está vivo (PIDs {vivos}); não dispare de novo'

os.makedirs(f'{PASTA_DRIVE}/logs', exist_ok=True)
with open(LOG, 'a', encoding='utf-8') as log:
    proc = subprocess.Popen(
        [sys.executable, '-m', 'dengue_sinan.runner', '--config', config_sessao],
        cwd=PASTA_REPO, stdout=log, stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL, start_new_session=True,
    )
print('runner iniciado, PID', proc.pid, '| log:', LOG)

In [ ]:
# Célula curta de acompanhamento: lê o relatório da sessão e o fim do log.
import json
from pathlib import Path

relatorio = runner.caminho_relatorio(load_config(config_sessao))
if relatorio.exists():
    dados = json.loads(relatorio.read_text(encoding='utf-8'))
    for bloco in dados['blocos']:
        print(bloco['bloco'], 'passou =', bloco['passou'])
    print('sessao_completa =', dados['sessao_completa'], '| passou =', dados['passou'])
else:
    print('relatório ainda não existe')
print(Path(LOG).read_text(encoding='utf-8')[-1500:])